# RF



## Data Set 1

In [4]:
# =============================================================================
# INSTALLING PACKAGES
# =============================================================================
# !pip install statsmodels optuna tabulate gdown

# =============================================================================
# IMPORTS
# =============================================================================
!pip install optuna
# Standard libraries
import os
import sys
import warnings
warnings.filterwarnings("ignore")

# Data manipulation
import numpy as np
import pandas as pd
from collections import Counter

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from matplotlib.colors import ListedColormap

# Scikit-learn: preprocessing
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn import model_selection, metrics, preprocessing

# Scikit-learn: model selection
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV,
    TimeSeriesSplit
)

# Scikit-learn: models & metrics
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

# Statistical analysis
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import pointbiserialr, chi2_contingency, spearmanr, entropy
from statsmodels.graphics.gofplots import qqplot

# Tabulate
from tabulate import tabulate

# Optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  # Suppress Optuna trial logs

# Google Drive download
import gdown

# Global random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


# =============================================================================
# LOADING THE DATA SET
# =============================================================================
# Dataset source:
# https://drive.google.com/file/d/1Z_KsoIumw-fvivVombIoWuRo0LOe2nCb/view?usp=sharing
# https://drive.google.com/file/d/1aD1PXfwEEZ_F2lQgxuPfj-TVbxQ6NajK/view?usp=sharing

file_id = "1qhCB1Dod6jeEsuKy5sNk4MNHSjtENSrx"
download_url = f"https://drive.google.com/uc?id={file_id}"

# Read the data
df = pd.read_csv(download_url)

# Backup original DataFrame
df_backup = df.copy()

In [5]:
df

,WS10M_lag1,PREC,RH,MIN_TEMP,MAX_TEMP,WD_sin,SURF_PRESSSURE,WD_cos,AVG_TEMP,WS10M_lag2,PREC_lag1,SL_PRESSURE,YEAR,MO,DY,WS10M
0,4.25,13.42,86.26,23.42,28.38,0.439939,99.68,0.898028,26.4,3.99,1.09,1011.0,2013,1,3,4.75
1,4.75,8.79,86.31,22.93,27.58,0.424199,99.65,0.905569,25.1,4.25,13.42,1010.1,2013,1,4,5.74
2,5.74,2.60,86.88,22.17,26.16,0.563526,99.50,0.826098,26.9,4.75,8.79,1007.5,2013,1,5,5.79
3,5.79,1.65,88.09,23.47,27.75,0.460200,99.47,0.887815,27.3,5.74,2.60,1008.2,2013,1,6,4.52
4,4.52,27.41,93.04,24.04,26.70,0.368125,99.46,0.929776,25.9,5.79,1.65,1009.2,2013,1,7,4.66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4010,6.01,9.45,88.57,24.40,30.35,0.381070,99.66,0.924546,29.5,4.29,3.50,1011.0,2023,12,27,6.43
4011,6.43,13.93,90.59,25.51,29.37,0.634731,99.58,0.772734,28.5,6.01,9.45,1010.9,2023,12,28,3.93
4012,3.93,9.80,89.03,25.03,29.30,0.689620,99.69,0.724172,28.2,6.43,13.93,1011.4,2023,12,29,2.59
4013,2.59,9.98,87.47,24.71,29.54,0.564967,99.67,0.825113,28.4,3.93,9.80,1010.9,2023,12,30,4.59


In [6]:

# =============================================================================
# DATA PREPROCESSING
# =============================================================================

# Create datetime directly from differently named columns
df['Date'] = pd.to_datetime({
    'year': df['YEAR'],
    'month': df['MO'],
    'day': df['DY']
})

# Set Date as the index
df = df.set_index('Date')

# Drop unnecessary columns
df = df.drop(columns=["YEAR", "MO", "DY"])

# Display the DataFrame
print(df.head())
print("\nColumns:", df.columns.tolist())



            WS10M_lag1   PREC     RH  MIN_TEMP  MAX_TEMP    WD_sin  \
Date                                                                 
2013-01-03        4.25  13.42  86.26     23.42     28.38  0.439939   
2013-01-04        4.75   8.79  86.31     22.93     27.58  0.424199   
2013-01-05        5.74   2.60  86.88     22.17     26.16  0.563526   
2013-01-06        5.79   1.65  88.09     23.47     27.75  0.460200   
2013-01-07        4.52  27.41  93.04     24.04     26.70  0.368125   

            SURF_PRESSSURE    WD_cos  AVG_TEMP  WS10M_lag2  PREC_lag1  \
Date                                                                    
2013-01-03           99.68  0.898028      26.4        3.99       1.09   
2013-01-04           99.65  0.905569      25.1        4.25      13.42   
2013-01-05           99.50  0.826098      26.9        4.75       8.79   
2013-01-06           99.47  0.887815      27.3        5.74       2.60   
2013-01-07           99.46  0.929776      25.9        5.79       1.65  

In [7]:
# =============================================================================
# TRAIN / TEST SPLIT  (80% train — 20% test, temporal order preserved)
# =============================================================================
## Data Sets

# Define features and target
X = df.drop(columns=['WS10M'])   # Features (all columns except target)
y = df['WS10M']                  # Target variable (wind speed)

numerical_cols = df.columns

# Define the split index — 80% for training, 20% for testing
split_index = int(len(X) * 0.8)

# Training set
X_train = X.iloc[:split_index]
y_train = y.iloc[:split_index]

# Testing set
X_test = X.iloc[split_index:]
y_test = y.iloc[split_index:]

print("\nX_train:\n", X_train.head())
print("y_train:\n", y_train.head())
print("X_test:\n",  X_test.head())
print("y_test:\n",  y_test.head())

print("\nShapes:")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape: ", X_test.shape)
print("y_test shape: ", y_test.shape)




X_train:
             WS10M_lag1   PREC     RH  MIN_TEMP  MAX_TEMP    WD_sin  \
Date                                                                 
2013-01-03        4.25  13.42  86.26     23.42     28.38  0.439939   
2013-01-04        4.75   8.79  86.31     22.93     27.58  0.424199   
2013-01-05        5.74   2.60  86.88     22.17     26.16  0.563526   
2013-01-06        5.79   1.65  88.09     23.47     27.75  0.460200   
2013-01-07        4.52  27.41  93.04     24.04     26.70  0.368125   

            SURF_PRESSSURE    WD_cos  AVG_TEMP  WS10M_lag2  PREC_lag1  \
Date                                                                    
2013-01-03           99.68  0.898028      26.4        3.99       1.09   
2013-01-04           99.65  0.905569      25.1        4.25      13.42   
2013-01-05           99.50  0.826098      26.9        4.75       8.79   
2013-01-06           99.47  0.887815      27.3        5.74       2.60   
2013-01-07           99.46  0.929776      25.9        5.79  

In [8]:
# =============================================================================
# SECTION 1 — ORIGINAL MODEL  (No Hyperparameter Optimisation)
# TimeSeriesSplit cross-validation applied on training data only
# =============================================================================
### Original

# Initialize the model with fixed random seed
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED)

# Initialize TimeSeriesSplit with 5 folds — applied on training data only
tscv = TimeSeriesSplit(n_splits=5)

# Lists to store evaluation metrics for each fold
train_mse_list,  test_mse_list  = [], []
train_mae_list,  test_mae_list  = [], []
train_rmse_list, test_rmse_list = [], []
train_mape_list, test_mape_list = [], []
train_r2_list,   test_r2_list   = [], []

# Time series cross-validation loop on training data
# NOTE: X_fold_train / X_fold_test are CV fold splits of X_train only.
#       They do NOT overwrite the original 80/20 X_train / X_test splits.
for fold_train_idx, fold_test_idx in tscv.split(X_train):
    # Split training data into fold train and fold validation sets
    X_fold_train = X_train.iloc[fold_train_idx]
    X_fold_val   = X_train.iloc[fold_test_idx]
    y_fold_train = y_train.iloc[fold_train_idx]
    y_fold_val   = y_train.iloc[fold_test_idx]

    # Train the model on the fold training data
    rf.fit(X_fold_train, y_fold_train)

    # Make predictions
    y_fold_train_pred = rf.predict(X_fold_train)
    y_fold_val_pred   = rf.predict(X_fold_val)

    # Calculate evaluation metrics
    fold_train_mse  = mean_squared_error(y_fold_train, y_fold_train_pred)
    fold_val_mse    = mean_squared_error(y_fold_val,   y_fold_val_pred)

    fold_train_mae  = mean_absolute_error(y_fold_train, y_fold_train_pred)
    fold_val_mae    = mean_absolute_error(y_fold_val,   y_fold_val_pred)

    fold_train_rmse = np.sqrt(fold_train_mse)
    fold_val_rmse   = np.sqrt(fold_val_mse)

    fold_train_mape = np.mean(np.abs((y_fold_train - y_fold_train_pred) / np.maximum(y_fold_train, 1e-8))) * 100
    fold_val_mape   = np.mean(np.abs((y_fold_val   - y_fold_val_pred)   / np.maximum(y_fold_val,   1e-8))) * 100

    fold_train_r2   = r2_score(y_fold_train, y_fold_train_pred)
    fold_val_r2     = r2_score(y_fold_val,   y_fold_val_pred)

    # Append results for each fold
    train_mse_list.append(fold_train_mse);   test_mse_list.append(fold_val_mse)
    train_mae_list.append(fold_train_mae);   test_mae_list.append(fold_val_mae)
    train_rmse_list.append(fold_train_rmse); test_rmse_list.append(fold_val_rmse)
    train_mape_list.append(fold_train_mape); test_mape_list.append(fold_val_mape)
    train_r2_list.append(fold_train_r2);     test_r2_list.append(fold_val_r2)

# Calculate average metrics over all CV folds
Original_CV_Train_MSE  = np.mean(train_mse_list)
Original_CV_Test_MSE   = np.mean(test_mse_list)
Original_CV_Train_MAE  = np.mean(train_mae_list)
Original_CV_Test_MAE   = np.mean(test_mae_list)
Original_CV_Train_RMSE = np.mean(train_rmse_list)
Original_CV_Test_RMSE  = np.mean(test_rmse_list)
Original_CV_Train_MAPE = np.mean(train_mape_list)
Original_CV_Test_MAPE  = np.mean(test_mape_list)
Original_CV_Train_R2   = np.mean(train_r2_list)
Original_CV_Test_R2    = np.mean(test_r2_list)

print(f"\n📊 Original Model — Cross-Validated Average Metrics (on training data folds):")
print(f"Train MSE:  {Original_CV_Train_MSE:.4f}, Val MSE:  {Original_CV_Test_MSE:.4f}")
print(f"Train MAE:  {Original_CV_Train_MAE:.4f}, Val MAE:  {Original_CV_Test_MAE:.4f}")
print(f"Train RMSE: {Original_CV_Train_RMSE:.4f}, Val RMSE: {Original_CV_Test_RMSE:.4f}")
print(f"Train MAPE: {Original_CV_Train_MAPE:.2f}%, Val MAPE: {Original_CV_Test_MAPE:.2f}%")
print(f"Train R²:   {Original_CV_Train_R2:.4f}, Val R²:   {Original_CV_Test_R2:.4f}")

# --- Optimal performance: refit on full X_train, evaluate on held-out X_test ---
rf_original_final = RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED)
rf_original_final.fit(X_train, y_train)

y_train_pred_original = rf_original_final.predict(X_train)
y_test_pred_original  = rf_original_final.predict(X_test)

Original_Train_MSE  = mean_squared_error(y_train,  y_train_pred_original)
Original_Test_MSE   = mean_squared_error(y_test,   y_test_pred_original)
Original_Train_MAE  = mean_absolute_error(y_train, y_train_pred_original)
Original_Test_MAE   = mean_absolute_error(y_test,  y_test_pred_original)
Original_Train_RMSE = np.sqrt(Original_Train_MSE)
Original_Test_RMSE  = np.sqrt(Original_Test_MSE)
Original_Train_MAPE = np.mean(np.abs((y_train - y_train_pred_original) / np.maximum(y_train, 1e-8))) * 100
Original_Test_MAPE  = np.mean(np.abs((y_test  - y_test_pred_original)  / np.maximum(y_test,  1e-8))) * 100
Original_Train_R2   = r2_score(y_train, y_train_pred_original)
Original_Test_R2    = r2_score(y_test,  y_test_pred_original)

print(f"\n📊 Original Model — Optimal Performance (full train → held-out test):")
print(f"Train MSE:  {Original_Train_MSE:.4f}, Test MSE:  {Original_Test_MSE:.4f}")
print(f"Train MAE:  {Original_Train_MAE:.4f}, Test MAE:  {Original_Test_MAE:.4f}")
print(f"Train RMSE: {Original_Train_RMSE:.4f}, Test RMSE: {Original_Test_RMSE:.4f}")
print(f"Train MAPE: {Original_Train_MAPE:.2f}%, Test MAPE: {Original_Test_MAPE:.2f}%")
print(f"Train R²:   {Original_Train_R2:.4f}, Test R²:   {Original_Test_R2:.4f}")

# Plot — Original model (last CV fold for illustration)
train_dates = X_train.index
test_dates  = X_test.index

fig = go.Figure()
fig.add_trace(go.Scatter(x=train_dates, y=y_train,              mode='lines', name='Y Train',           line=dict(color='blue')))
fig.add_trace(go.Scatter(x=test_dates,  y=y_test,               mode='lines', name='Y Test',            line=dict(color='green')))
fig.add_trace(go.Scatter(x=train_dates, y=y_train_pred_original, mode='lines', name='Y Train Predicted', line=dict(color='orange', dash='dot')))
fig.add_trace(go.Scatter(x=test_dates,  y=y_test_pred_original,  mode='lines', name='Y Test Predicted',  line=dict(color='red',    dash='dot')))
fig.update_layout(
    title="Original Model — Actual vs Predicted Values",
    xaxis_title="Date",
    yaxis_title="Wind Speed (WS10M)",
    template="plotly_white"
)
fig.show()





📊 Original Model — Cross-Validated Average Metrics (on training data folds):
Train MSE:  0.0680, Val MSE:  0.5719
Train MAE:  0.2005, Val MAE:  0.5737
Train RMSE: 0.2607, Val RMSE: 0.7561
Train MAPE: 5.76%, Val MAPE: 16.35%
Train R²:   0.9722, Val R²:   0.7666

📊 Original Model — Optimal Performance (full train → held-out test):
Train MSE:  0.0718, Test MSE:  0.5843
Train MAE:  0.2026, Test MAE:  0.5805
Train RMSE: 0.2680, Test RMSE: 0.7644
Train MAPE: 5.70%, Test MAPE: 15.85%
Train R²:   0.9715, Test R²:   0.7577


In [9]:
# =============================================================================
# SECTION 2 — GRID SEARCH CV  Hyperparameter Tuning
# TimeSeriesSplit cross-validation applied on training data only
# =============================================================================
### Grid Search CV Hyper Parameter Tuning

# Define parameter grid (bootstrap removed)
param_grid = {
    'n_estimators':      [100, 150, 200],
    'max_depth':         [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', None]   # Valid sklearn options only
}

# Initialize RandomForestRegressor
rf_grid = RandomForestRegressor(random_state=RANDOM_SEED)

# TimeSeriesSplit ensures training always precedes validation in time
tscv = TimeSeriesSplit(n_splits=5)

# Initialize GridSearchCV — fitted on X_train only (no test set exposure)
grid_search = GridSearchCV(
    estimator=rf_grid,
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=2,
    refit=True          # Refit best model on full X_train automatically
)

# Fit the model with GridSearchCV on training data only
grid_search.fit(X_train, y_train)

# Extract best parameters and best model
grid_best_params = grid_search.best_params_
grid_best_score  = grid_search.best_score_
best_rf_grid     = grid_search.best_estimator_

print(f"\n✅ Grid Search — Best Hyperparameters: {grid_best_params}")
print(f"📉 Best CV Score (Negative MSE): {grid_best_score:.4f}")

# --- Optimal performance: predict using best model on full X_train and X_test ---
y_train_pred_grid = best_rf_grid.predict(X_train)
y_test_pred_grid  = best_rf_grid.predict(X_test)

# Evaluate performance
grid_train_mse  = mean_squared_error(y_train,  y_train_pred_grid)
grid_test_mse   = mean_squared_error(y_test,   y_test_pred_grid)
grid_train_mae  = mean_absolute_error(y_train, y_train_pred_grid)
grid_test_mae   = mean_absolute_error(y_test,  y_test_pred_grid)
grid_train_rmse = np.sqrt(grid_train_mse)
grid_test_rmse  = np.sqrt(grid_test_mse)
grid_train_mape = np.mean(np.abs((y_train - y_train_pred_grid) / np.maximum(y_train, 1e-8))) * 100
grid_test_mape  = np.mean(np.abs((y_test  - y_test_pred_grid)  / np.maximum(y_test,  1e-8))) * 100
grid_train_r2   = r2_score(y_train, y_train_pred_grid)
grid_test_r2    = r2_score(y_test,  y_test_pred_grid)

# Handle division by zero in MAPE — already handled via np.maximum above
print(f"\n📊 Grid Search — Optimal Performance:")
print(f"Train MSE:  {grid_train_mse:.4f}, Test MSE:  {grid_test_mse:.4f}")
print(f"Train MAE:  {grid_train_mae:.4f}, Test MAE:  {grid_test_mae:.4f}")
print(f"Train RMSE: {grid_train_rmse:.4f}, Test RMSE: {grid_test_rmse:.4f}")
print(f"Train MAPE: {grid_train_mape:.2f}%, Test MAPE: {grid_test_mape:.2f}%")
print(f"Train R²:   {grid_train_r2:.4f}, Test R²:   {grid_test_r2:.4f}")

# Plot — Grid Search best model
fig = go.Figure()
fig.add_trace(go.Scatter(x=train_dates, y=y_train,            mode='lines', name='Y Train',           line=dict(color='blue')))
fig.add_trace(go.Scatter(x=test_dates,  y=y_test,             mode='lines', name='Y Test',            line=dict(color='green')))
fig.add_trace(go.Scatter(x=train_dates, y=y_train_pred_grid,  mode='lines', name='Y Train Predicted', line=dict(color='orange', dash='dot')))
fig.add_trace(go.Scatter(x=test_dates,  y=y_test_pred_grid,   mode='lines', name='Y Test Predicted',  line=dict(color='red',    dash='dot')))
fig.update_layout(
    title="Grid Search CV — Actual vs Predicted Values",
    xaxis_title="Date",
    yaxis_title="Wind Speed (WS10M)",
    template="plotly_white"
)
fig.show()




Fitting 5 folds for each of 243 candidates, totalling 1215 fits

✅ Grid Search — Best Hyperparameters: {'max_depth': 20, 'max_features': None, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 200}
📉 Best CV Score (Negative MSE): -0.5608

📊 Grid Search — Optimal Performance:
Train MSE:  0.1724, Test MSE:  0.5821
Train MAE:  0.3046, Test MAE:  0.5800
Train RMSE: 0.4152, Test RMSE: 0.7630
Train MAPE: 8.58%, Test MAPE: 15.87%
Train R²:   0.9316, Test R²:   0.7586


In [10]:
# =============================================================================
# SECTION 3 — RANDOMIZED SEARCH CV  Hyperparameter Tuning
# TimeSeriesSplit cross-validation applied on training data only
# =============================================================================
### Random Search CV

# Define the Random Forest Regressor model with fixed random seed
rf_random = RandomForestRegressor(random_state=RANDOM_SEED)

# Define the parameter grid for hyperparameter tuning
# NOTE: 'auto' has been removed — deprecated in recent sklearn versions; 'sqrt' is equivalent
param_grid_random = {
    'n_estimators':      [50, 100, 150, 200],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 7, 10],
    'min_samples_leaf':  [1, 2, 4, 7],
    'max_features':      ['sqrt', 'log2', None]   # 'auto' removed (deprecated)
}

# Set up the TimeSeriesSplit object for time series cross-validation
tscv = TimeSeriesSplit(n_splits=5)

# Set up RandomizedSearchCV with random_state and TimeSeriesSplit
# Fitted on X_train only (no test set exposure)
random_search = RandomizedSearchCV(
    estimator=rf_random,
    param_distributions=param_grid_random,
    n_iter=100,
    cv=tscv,
    n_jobs=-1,
    verbose=2,
    scoring='neg_mean_squared_error',
    random_state=RANDOM_SEED,    # Fixed random seed
    refit=True                   # Refit best model on full X_train automatically
)

# Fit the model on training data only
random_search.fit(X_train, y_train)

# Get the best parameters and best estimator
random_best_params = random_search.best_params_
random_best_rf     = random_search.best_estimator_

print(f"\n✅ Randomized Search — Best Parameters: {random_best_params}")

# --- Optimal performance: predict using best model on full X_train and X_test ---
y_train_pred_random = random_best_rf.predict(X_train)
y_test_pred_random  = random_best_rf.predict(X_test)

# Evaluate performance
random_train_mse  = mean_squared_error(y_train,  y_train_pred_random)
random_test_mse   = mean_squared_error(y_test,   y_test_pred_random)
random_train_mae  = mean_absolute_error(y_train, y_train_pred_random)
random_test_mae   = mean_absolute_error(y_test,  y_test_pred_random)
random_train_rmse = np.sqrt(random_train_mse)
random_test_rmse  = np.sqrt(random_test_mse)
random_train_mape = np.mean(np.abs((y_train - y_train_pred_random) / np.maximum(y_train, 1e-8))) * 100
random_test_mape  = np.mean(np.abs((y_test  - y_test_pred_random)  / np.maximum(y_test,  1e-8))) * 100
random_train_r2   = r2_score(y_train, y_train_pred_random)
random_test_r2    = r2_score(y_test,  y_test_pred_random)

print(f"\n📊 Randomized Search — Optimal Performance:")
print(f"Train MSE:  {random_train_mse:.4f}, Test MSE:  {random_test_mse:.4f}")
print(f"Train MAE:  {random_train_mae:.4f}, Test MAE:  {random_test_mae:.4f}")
print(f"Train RMSE: {random_train_rmse:.4f}, Test RMSE: {random_test_rmse:.4f}")
print(f"Train MAPE: {random_train_mape:.2f}%, Test MAPE: {random_test_mape:.2f}%")
print(f"Train R²:   {random_train_r2:.4f}, Test R²:   {random_test_r2:.4f}")

# Plot — Randomized Search best model
fig = go.Figure()
fig.add_trace(go.Scatter(x=train_dates, y=y_train,              mode='lines', name='Y Train',           line=dict(color='blue')))
fig.add_trace(go.Scatter(x=test_dates,  y=y_test,               mode='lines', name='Y Test',            line=dict(color='green')))
fig.add_trace(go.Scatter(x=train_dates, y=y_train_pred_random,  mode='lines', name='Y Train Predicted', line=dict(color='orange', dash='dot')))
fig.add_trace(go.Scatter(x=test_dates,  y=y_test_pred_random,   mode='lines', name='Y Test Predicted',  line=dict(color='red',    dash='dot')))
fig.update_layout(
    title="Randomized Search CV — Actual vs Predicted Values",
    xaxis_title="Date",
    yaxis_title="Wind Speed (WS10M)",
    template="plotly_white"
)
fig.show()



Fitting 5 folds for each of 100 candidates, totalling 500 fits

✅ Randomized Search — Best Parameters: {'n_estimators': 200, 'min_samples_split': 7, 'min_samples_leaf': 4, 'max_features': None, 'max_depth': 10}

📊 Randomized Search — Optimal Performance:
Train MSE:  0.2280, Test MSE:  0.5865
Train MAE:  0.3629, Test MAE:  0.5841
Train RMSE: 0.4775, Test RMSE: 0.7658
Train MAPE: 10.08%, Test MAPE: 16.00%
Train R²:   0.9095, Test R²:   0.7568


In [11]:
# =============================================================================
# SECTION 4 — OPTUNA  Hyperparameter Tuning
# FIX: TimeSeriesSplit CV on X_train inside the objective function.
# Original code used X_test directly in the objective — this was data leakage
# because hyperparameters were being selected based on held-out test performance.
# =============================================================================
### Optuna

# Define the objective function for optimisation
def objective(trial):
    # Optimised parameter space
    n_estimators       = trial.suggest_int('n_estimators',       50,  200, step=50)
    max_depth          = trial.suggest_categorical('max_depth',   [None, 10, 20, 30])
    min_samples_split  = trial.suggest_int('min_samples_split',  2,   10)
    min_samples_leaf   = trial.suggest_int('min_samples_leaf',   1,   7)
    max_features       = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])

    # Define the model with the suggested hyperparameters
    rf_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=RANDOM_SEED
    )

    # Use TimeSeriesSplit CV on X_train only — avoids any exposure to held-out X_test
    # FIX: previously X_test was used directly here, causing data leakage
    tscv_optuna = TimeSeriesSplit(n_splits=5)
    fold_mse_scores = []

    for fold_train_idx, fold_val_idx in tscv_optuna.split(X_train):
        X_fold_tr  = X_train.iloc[fold_train_idx]
        X_fold_val = X_train.iloc[fold_val_idx]
        y_fold_tr  = y_train.iloc[fold_train_idx]
        y_fold_val = y_train.iloc[fold_val_idx]

        rf_model.fit(X_fold_tr, y_fold_tr)
        y_fold_val_pred = rf_model.predict(X_fold_val)
        fold_mse_scores.append(mean_squared_error(y_fold_val, y_fold_val_pred))

    # Return the mean CV MSE across all folds — no test set exposure
    return np.mean(fold_mse_scores)

# Set up the Optuna study using TPE Sampler
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED)
)
study.optimize(objective, n_trials=50)

# Get the best parameters found by Optuna
optuna_best_params = study.best_params
print(f"\n✅ Best Parameters Found by Optuna:\n{optuna_best_params}")

# --- Optimal performance: refit on full X_train, evaluate on X_train and X_test ---
best_rf_optuna = RandomForestRegressor(**optuna_best_params, random_state=RANDOM_SEED)
best_rf_optuna.fit(X_train, y_train)

# Make predictions
y_train_pred_optuna = best_rf_optuna.predict(X_train)
y_test_pred_optuna  = best_rf_optuna.predict(X_test)

# Evaluate performance
train_mse_optuna  = mean_squared_error(y_train,  y_train_pred_optuna)
test_mse_optuna   = mean_squared_error(y_test,   y_test_pred_optuna)
train_mae_optuna  = mean_absolute_error(y_train, y_train_pred_optuna)
test_mae_optuna   = mean_absolute_error(y_test,  y_test_pred_optuna)
train_rmse_optuna = np.sqrt(train_mse_optuna)
test_rmse_optuna  = np.sqrt(test_mse_optuna)
train_r2_optuna   = r2_score(y_train, y_train_pred_optuna)
test_r2_optuna    = r2_score(y_test,  y_test_pred_optuna)

# Avoid division by zero in MAPE
train_mape_optuna = np.mean(np.abs((y_train - y_train_pred_optuna) / np.maximum(y_train, 1e-8))) * 100
test_mape_optuna  = np.mean(np.abs((y_test  - y_test_pred_optuna)  / np.maximum(y_test,  1e-8))) * 100

print(f"\n📊 Optuna — Optimal Performance:")
print(f"Train MSE:  {train_mse_optuna:.4f}, Test MSE:  {test_mse_optuna:.4f}")
print(f"Train MAE:  {train_mae_optuna:.4f}, Test MAE:  {test_mae_optuna:.4f}")
print(f"Train RMSE: {train_rmse_optuna:.4f}, Test RMSE: {test_rmse_optuna:.4f}")
print(f"Train MAPE: {train_mape_optuna:.2f}%, Test MAPE: {test_mape_optuna:.2f}%")
print(f"Train R²:   {train_r2_optuna:.4f}, Test R²:   {test_r2_optuna:.4f}")

# Optuna trials summary table
optuna_results     = study.trials_dataframe()
cols_to_extract    = [
    'number', 'value',
    'params_n_estimators', 'params_max_depth',
    'params_min_samples_split', 'params_min_samples_leaf', 'params_max_features'
]
optuna_table_df    = optuna_results[cols_to_extract].copy()
optuna_table_df.columns = [
    'Trial', 'CV MSE',
    'n_estimators', 'max_depth',
    'min_samples_split', 'min_samples_leaf', 'max_features'
]

# Create Plotly Table for Optuna results
fig_table_optuna = go.Figure(data=[go.Table(
    header=dict(values=list(optuna_table_df.columns),
                fill_color='paleturquoise',
                align='left'),
    cells=dict(values=[optuna_table_df[col] for col in optuna_table_df.columns],
               fill_color='lavender',
               align='left')
)])
fig_table_optuna.update_layout(
    title="Optuna Results — Random Forest Hyperparameter Tuning",
    height=600,
    template='plotly_white'
)
fig_table_optuna.show()

# Plot — Optuna best model
fig = go.Figure()
fig.add_trace(go.Scatter(x=train_dates, y=y_train,             mode='lines', name='Y Train',           line=dict(color='blue')))
fig.add_trace(go.Scatter(x=test_dates,  y=y_test,              mode='lines', name='Y Test',            line=dict(color='green')))
fig.add_trace(go.Scatter(x=train_dates, y=y_train_pred_optuna, mode='lines', name='Y Train Predicted', line=dict(color='orange', dash='dot')))
fig.add_trace(go.Scatter(x=test_dates,  y=y_test_pred_optuna,  mode='lines', name='Y Test Predicted',  line=dict(color='red',    dash='dot')))
fig.update_layout(
    title="Optuna — Actual vs Predicted Values",
    xaxis_title="Date",
    yaxis_title="Wind Speed (WS10M)",
    template="plotly_white"
)
fig.show()




✅ Best Parameters Found by Optuna:
{'n_estimators': 200, 'max_depth': None, 'min_samples_split': 3, 'min_samples_leaf': 5, 'max_features': None}

📊 Optuna — Optimal Performance:
Train MSE:  0.2038, Test MSE:  0.5848
Train MAE:  0.3345, Test MAE:  0.5823
Train RMSE: 0.4515, Test RMSE: 0.7647
Train MAPE: 9.41%, Test MAPE: 15.92%
Train R²:   0.9191, Test R²:   0.7575


In [12]:

# =============================================================================
# FINAL COMPARISON TABLE
# Optimal train/test performance for each method
# (best model refitted on full X_train, evaluated on held-out X_test)
# =============================================================================

results = {
    "Method": ["Original Model", "Grid Search", "Randomized Search", "Optuna"],

    "Train MSE":    [Original_Train_MSE,  grid_train_mse,  random_train_mse,  train_mse_optuna],
    "Test MSE":     [Original_Test_MSE,   grid_test_mse,   random_test_mse,   test_mse_optuna],

    "Train MAE":    [Original_Train_MAE,  grid_train_mae,  random_train_mae,  train_mae_optuna],
    "Test MAE":     [Original_Test_MAE,   grid_test_mae,   random_test_mae,   test_mae_optuna],

    "Train RMSE":   [Original_Train_RMSE, grid_train_rmse, random_train_rmse, train_rmse_optuna],
    "Test RMSE":    [Original_Test_RMSE,  grid_test_rmse,  random_test_rmse,  test_rmse_optuna],

    "Train MAPE (%)": [Original_Train_MAPE, grid_train_mape, random_train_mape, train_mape_optuna],
    "Test MAPE (%)":  [Original_Test_MAPE,  grid_test_mape,  random_test_mape,  test_mape_optuna],

    "Train R²":     [Original_Train_R2,   grid_train_r2,   random_train_r2,   train_r2_optuna],
    "Test R²":      [Original_Test_R2,    grid_test_r2,    random_test_r2,    test_r2_optuna],
}

# Convert dictionary to Pandas DataFrame
df_results = pd.DataFrame(results)

# Print the title
print("\n" + "=" * 60)
print("   Variable Set 1 — Random Forest Modelling")
print("   (Optimal Train/Test Performance per Method)")
print("=" * 60 + "\n")

# Display the table
print(df_results.to_string(index=False))

# Save the DataFrame to a CSV file
df_results.to_csv("variableSet_1_random_forest_results.csv", index=False)
print("\nResults saved to 'variableSet_1_random_forest_results.csv'")



   Variable Set 1 — Random Forest Modelling
   (Optimal Train/Test Performance per Method)

           Method  Train MSE  Test MSE  Train MAE  Test MAE  Train RMSE  Test RMSE  Train MAPE (%)  Test MAPE (%)  Train R²  Test R²
   Original Model   0.071837  0.584321   0.202551  0.580509    0.268025   0.764409        5.703723      15.849229  0.971485 0.757688
      Grid Search   0.172377  0.582145   0.304623  0.579983    0.415183   0.762984        8.576856      15.866937  0.931577 0.758591
Randomized Search   0.228000  0.586458   0.362892  0.584118    0.477494   0.765805       10.078571      15.998774  0.909499 0.756802
           Optuna   0.203814  0.584759   0.334457  0.582314    0.451457   0.764695        9.409737      15.915087  0.919099 0.757507

Results saved to 'variableSet_1_random_forest_results.csv'
